# nb42 — Data request mockup: per-cell signal-energy truth

The current ntuples (`clusters_matched`) give us, per matched cluster, the jagged per-cell branches `cell_x`, `cell_y`, `energy`, `cell_energies_front/back`, `cell_times_front/back` — the *mixed* photon+pileup deposit, with no way to tell how much of each cell belongs to the matched photon. We request **one added per-cell branch pair: `cell_energies_signal_front` and `cell_energies_signal_back` — the energy deposited in that cell by the matched signal photon, same jagged shape as `cell_energies_front`** (equivalently a single `cell_signal_fraction`). This enables per-cell supervision of the pileup/photon decomposition, the exact recipe Belle II used to gain ~30% resolution under beam background (arXiv:2306.04179); nb26–nb31 showed this decomposition cannot be learned from mixed events alone.

In [2]:
import os, glob
import numpy as np
import uproot
import awkward as ak
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path

pio.renderers.default = "plotly_mimetype"
pio.templates.default = "plotly_white"
rng = np.random.default_rng(42)

REPO = Path(os.environ.get("REPO_DIR", ".."))
OUT = REPO / "reports" / "figures" / "interactive"
OUT.mkdir(parents=True, exist_ok=True)

mb_file = sorted(glob.glob(str(REPO / "data" / "minimum_bias" / "*" / "*.root")))[0]
mb_tree = uproot.open(mb_file)["clusters_matched"]
mb = mb_tree.arrays(["event", "icell", "cell_x", "cell_y", "energy",
                     "cell_energies_front", "cell_energies_back",
                     "cell_times_front", "cell_times_back",
                     "seed_cell_x", "seed_cell_y"], entry_stop=50)
ncells = ak.num(mb["cell_x"])
ENTRY = int(np.argmax(np.asarray(ncells) >= 25))
ev = {k: np.asarray(mb[k][ENTRY]) for k in ["icell", "cell_x", "cell_y", "energy",
                                            "cell_energies_front", "cell_energies_back",
                                            "cell_times_front", "cell_times_back"]}
seed_x = float(mb["seed_cell_x"][ENTRY])
seed_y = float(mb["seed_cell_y"][ENTRY])
uy = np.sort(np.unique(ev["cell_y"]))
dy = np.diff(uy)
PITCH = float(np.median(dy[dy > 10]))
di = np.rint((ev["cell_x"] - seed_x) / PITCH).astype(int)
dj = np.rint((ev["cell_y"] - seed_y) / PITCH).astype(int)


def valid_time(t):
    return np.isfinite(t) & (t != 0) & (np.abs(t) < 1e4)


print(f"min-bias file: {Path(mb_file).name}")
print(f"entry {ENTRY}, event {int(mb['event'][ENTRY])}, n cells {len(ev['cell_x'])}")
print(f"pitch {PITCH:.2f} mm, seed ({seed_x:.1f}, {seed_y:.1f})")
print(f"cells in 9x9 window: {int(np.sum((np.abs(di) <= 4) & (np.abs(dj) <= 4)))}")

min-bias file: matched_1001_1250.root
entry 0, event 0, n cells 208
pitch 15.03 mm, seed (-1546.9, 159.3)
cells in 9x9 window: 0


**Fig 1 — the ask, in table form.** Left: eight real cells from one min-bias cluster exactly as the ntuple stores them today (MeV, ns). Right: the same rows with the two requested columns appended — `e_signal` (deposit from the matched photon) and `e_pileup` (the remainder); today both are `needed`.

In [3]:
win = (np.abs(di) <= 4) & (np.abs(dj) <= 4)
order = np.argsort(ev["energy"][win])[::-1][:8]
idx8 = np.where(win)[0][order]
rows = {
    "icell": [str(int(v)) for v in ev["icell"][idx8]],
    "cell_x": [f"{v:.1f}" for v in ev["cell_x"][idx8]],
    "cell_y": [f"{v:.1f}" for v in ev["cell_y"][idx8]],
    "energy": [f"{v:.1f}" for v in ev["energy"][idx8]],
    "E_front": [f"{v:.1f}" for v in ev["cell_energies_front"][idx8]],
    "E_back": [f"{v:.1f}" for v in ev["cell_energies_back"][idx8]],
    "t_front": [f"{v:.2f}" for v in ev["cell_times_front"][idx8]],
}
cols_now = list(rows.keys())
cols_req = cols_now + ["e_signal", "e_pileup"]
vals_now = [rows[c] for c in cols_now]
vals_req = [rows[c] for c in cols_now] + [["needed"] * 8, ["needed"] * 8]
hl = "#f6c344"
hl_cells = "#fdeecb"
head_req = ["#e8eef7"] * len(cols_now) + [hl, hl]
cell_req = [["white"] * 8] * len(cols_now) + [[hl_cells] * 8, [hl_cells] * 8]

fig1 = go.Figure()
fig1.add_trace(go.Table(
    domain=dict(x=[0.0, 0.44], y=[0, 0.92]),
    header=dict(values=[f"<b>{c}</b>" for c in cols_now], fill_color="#e8eef7",
                font=dict(size=11), align="center"),
    cells=dict(values=vals_now, font=dict(size=10), align="center", height=22)))
fig1.add_trace(go.Table(
    domain=dict(x=[0.50, 1.0], y=[0, 0.92]),
    header=dict(values=[f"<b>{c}</b>" for c in cols_req], fill_color=head_req,
                font=dict(size=11), align="center"),
    cells=dict(values=vals_req, fill_color=cell_req, font=dict(size=10),
               align="center", height=22)))
fig1.update_layout(
    height=380, margin=dict(l=10, r=10, t=60, b=10),
    annotations=[
        dict(x=0.19, y=1.0, xref="paper", yref="paper", showarrow=False,
             text="<b>current clusters_matched</b> (real min-bias cells, MeV / ns)",
             font=dict(size=13)),
        dict(x=0.78, y=1.0, xref="paper", yref="paper", showarrow=False,
             text="<b>requested</b>: + per-cell signal / pileup split",
             font=dict(size=13))])
fig1.write_html(OUT / "ifig6_request_tables.html", include_plotlyjs="cdn")
fig1

**Fig 2 — what we see now.** The 9×9 Chebyshev window (the analysis window) around the seed of that same real event: total per-cell energy (log color) and per-cell front time. The energy map is signal+pileup mixed; timing carries separation power, but with no per-cell truth we cannot supervise it. Grey = cell absent or invalid time.

In [4]:
def agg_window(dii, djj, e, t=None, half=4):
    E = {}
    T = {}
    Emax = {}
    for k in range(len(dii)):
        a, b = int(dii[k]), int(djj[k])
        if abs(a) > half or abs(b) > half:
            continue
        E[(a, b)] = E.get((a, b), 0.0) + float(e[k])
        if t is not None and float(e[k]) >= Emax.get((a, b), -1):
            Emax[(a, b)] = float(e[k])
            T[(a, b)] = float(t[k])
    return E, T


def grid_from(d, half=4, log=False, valid=None):
    z = np.full((2 * half + 1, 2 * half + 1), np.nan)
    for (a, b), v in d.items():
        if valid is not None and not valid(v):
            continue
        z[b + half, a + half] = np.log10(v) if log and v > 0 else (np.nan if log else v)
    return z


axes = list(range(-4, 5))
Ew, Tw = agg_window(di, dj, ev["energy"], ev["cell_times_front"])
zE = grid_from(Ew, log=True)
zT = grid_from(Tw, valid=lambda v: bool(valid_time(np.array([v]))[0]))
txtE = [[f"{10**v:.0f}" if np.isfinite(v) else "" for v in row] for row in zE]

grey = np.zeros((9, 9))
fig2 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                     subplot_titles=["total energy [MeV], log color", "front time [ns], valid only"])
for c in [1, 2]:
    fig2.add_trace(go.Heatmap(z=grey, x=axes, y=axes, colorscale=[[0, "#d9d9d9"], [1, "#d9d9d9"]],
                              showscale=False, hoverinfo="skip"), row=1, col=c)
fig2.add_trace(go.Heatmap(z=zE, x=axes, y=axes, colorscale="Viridis",
                          text=txtE, texttemplate="%{text}", textfont=dict(size=8),
                          colorbar=dict(title="log10 E", x=0.42, len=0.9)), row=1, col=1)
fig2.add_trace(go.Heatmap(z=zT, x=axes, y=axes, colorscale="RdBu",
                          colorbar=dict(title="t [ns]", x=1.0, len=0.9)), row=1, col=2)
for c in [1, 2]:
    fig2.update_xaxes(title_text="di", row=1, col=c, dtick=1)
    fig2.update_yaxes(title_text="dj", row=1, col=c, dtick=1)
fig2.update_layout(height=430, width=900, margin=dict(l=40, r=40, t=60, b=40),
                   title_text=f"now: one real min-bias window (entry {ENTRY})", title_x=0.5)
fig2

**Fig 3 — what we ask the simulation to store.** One synthetic overlay window built the nb28 way — a clean shower (pure signal) merged onto a min-bias patch far from its donor seed (pure pileup) — is the only place we currently *know* the decomposition. The requested branch would give exactly these three panels for every real event.

In [5]:
cl_file = sorted(glob.glob(str(REPO / "data" / "full" / "matched_*.root")))[0]
cl_tree = uproot.open(cl_file)["clusters_matched"]
cl_keys = set(cl_tree.keys())
want = ["cell_x", "cell_y", "energy", "cell_energies_front", "cell_energies_back", "cell_times_front"]
have = [k for k in want if k in cl_keys]
cl = cl_tree.arrays(have, entry_stop=50)
if "energy" in have:
    cl_n = ak.num(cl["energy"])
else:
    cl_n = ak.num(cl["cell_energies_front"])
CL_ENTRY = int(np.argmax(np.asarray(cl_n) >= 10))
clev = {k: np.asarray(cl[k][CL_ENTRY]) for k in have}
if "energy" not in clev:
    clev["energy"] = clev["cell_energies_front"] + clev["cell_energies_back"]
if "cell_times_front" not in clev:
    clev["cell_times_front"] = np.zeros_like(clev["energy"])
cuy = np.sort(np.unique(clev["cell_y"]))
cdy = np.diff(cuy)
CPITCH = float(np.median(cdy[cdy > 10])) if np.any(cdy > 10) else PITCH
cs = int(np.argmax(clev["energy"]))
cdi = np.rint((clev["cell_x"] - clev["cell_x"][cs]) / CPITCH).astype(int)
cdj = np.rint((clev["cell_y"] - clev["cell_y"][cs]) / CPITCH).astype(int)
Esig, Tsig = agg_window(cdi, cdj, clev["energy"], clev["cell_times_front"])

far = np.maximum(np.abs(di), np.abs(dj)) >= 6
fi = np.where(far)[0]
ci_k = fi[int(np.argmax(ev["energy"][fi]))]
ci, cj = int(di[ci_k]), int(dj[ci_k])
Epu, Tpu = agg_window(di - ci, dj - cj, ev["energy"], ev["cell_times_front"])

Etot = dict(Esig)
for k, v in Epu.items():
    Etot[k] = Etot.get(k, 0.0) + v

zmax = np.log10(max(Etot.values()))
fig3 = make_subplots(rows=1, cols=3, horizontal_spacing=0.06,
                     subplot_titles=["total = signal + pileup", "signal only", "pileup only"])
for c, d in enumerate([Etot, Esig, Epu], start=1):
    fig3.add_trace(go.Heatmap(z=grey, x=axes, y=axes,
                              colorscale=[[0, "#d9d9d9"], [1, "#d9d9d9"]],
                              showscale=False, hoverinfo="skip"), row=1, col=c)
    fig3.add_trace(go.Heatmap(z=grid_from(d, log=True), x=axes, y=axes,
                              colorscale="Viridis", zmin=0, zmax=zmax,
                              showscale=(c == 3),
                              colorbar=dict(title="log10 E", len=0.9)), row=1, col=c)
    fig3.update_xaxes(title_text="di", row=1, col=c, dtick=1)
    fig3.update_yaxes(dtick=1, row=1, col=c)
fig3.update_yaxes(title_text="dj", row=1, col=1)
fig3.update_layout(height=380, width=1050, margin=dict(l=40, r=40, t=70, b=40),
                   title_text="what per-cell truth looks like (here synthetic - we request this from full simulation)",
                   title_x=0.5)
fig3.write_html(OUT / "ifig7_request_grid.html", include_plotlyjs="cdn")
print(f"clean donor: {Path(cl_file).name} entry {CL_ENTRY}, {int(cl_n[CL_ENTRY])} cells; pileup patch center offset ({ci},{cj})")
fig3

clean donor: matched_1001_1010.root entry 1, 137 cells; pileup patch center offset (52,-4)


**Fig 4 — the label we request, seen in space-time.** Each marker is one cell of the merged synthetic window: position (di, dj), height = cell time (real times where valid, jittered mock otherwise — signal tight, pileup spread), size ~ log energy, color = signal fraction e_sig/(e_sig+e_pileup). This fraction, per cell, is the branch we are asking for.

In [6]:
ts_valid = np.array([v for v in Tsig.values() if valid_time(np.array([v]))[0]])
tp_valid = np.array([v for v in Tpu.values() if valid_time(np.array([v]))[0]])
t_sig0 = float(np.median(ts_valid)) if len(ts_valid) else 0.0
t_pu0 = float(np.median(tp_valid)) if len(tp_valid) else t_sig0 + 0.5

xs, ys, zs, ss, fs, hv = [], [], [], [], [], []
for k in sorted(Etot.keys()):
    es = Esig.get(k, 0.0)
    ep = Epu.get(k, 0.0)
    if es + ep <= 0:
        continue
    f = es / (es + ep)
    tsig = Tsig.get(k, np.nan)
    if not valid_time(np.array([tsig]))[0]:
        tsig = t_sig0 + rng.normal(0, 0.08)
    tpu = Tpu.get(k, np.nan)
    if not valid_time(np.array([tpu]))[0]:
        tpu = t_pu0 + rng.normal(0, 0.45)
    t = f * tsig + (1 - f) * tpu
    xs.append(k[0]); ys.append(k[1]); zs.append(t)
    ss.append(4 + 4 * np.log10(1 + es + ep))
    fs.append(f)
    hv.append(f"di={k[0]} dj={k[1]}<br>E={es+ep:.1f} MeV<br>f_signal={f:.2f}<br>t={t:.2f} ns")

fig4 = go.Figure(go.Scatter3d(
    x=xs, y=ys, z=zs, mode="markers",
    marker=dict(size=ss, color=fs, colorscale="Portland", cmin=0, cmax=1,
                colorbar=dict(title="f_signal = e_sig/(e_sig+e_pileup)"),
                opacity=0.9, line=dict(width=0.5, color="#444")),
    text=hv, hoverinfo="text"))
fig4.update_layout(height=560, width=760,
                   scene=dict(xaxis_title="di", yaxis_title="dj", zaxis_title="cell time [ns]"),
                   title_text="the label we request, seen in space-time", title_x=0.5,
                   margin=dict(l=10, r=10, t=50, b=10))
fig4.write_html(OUT / "ifig8_request_spacetime.html", include_plotlyjs="cdn")
fig4